# 14 · Security suite (attacks against a throwaway copy)

Starts its own GEETA on port 8099 with a **temporary database and outbox** — the real accounts and
children are never touched — then attacks it: cross-site requests, injection, oversized and
flooding requests, password / PIN / 2-step guessing, access control between parents, tutors, kids
and admins, file-path tricks, cookies and headers, the pilot tools, the email endpoints and the
security log. Fails if any attack gets through. Results: `outputs/tests/security.json`.
Takes ~4 minutes (it waits out two rate limits).

In [ ]:
# ---- throwaway GEETA server: own temp database + outbox, port 8099 (never the real data) ----
import os, re, sys, json, time, signal, struct, zlib, socket, tempfile, subprocess, pathlib
import urllib.request, urllib.error, http.cookiejar

ROOT = pathlib.Path.cwd()
while not (ROOT / "PROJECT_BRIEF.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
PORT = 8099
B = f"http://127.0.0.1:{PORT}"
TMP = pathlib.Path(tempfile.mkdtemp(prefix="geeta_test_"))
subprocess.run(f"lsof -ti tcp:{PORT} | xargs kill 2>/dev/null", shell=True)
time.sleep(1)
_env = dict(os.environ, GEETA_PORTAL_DIR=str(TMP / "portal"), GEETA_PORT=str(PORT), GEETA_OUTBOX_DIR=str(TMP / "outbox"))
SERVER = subprocess.Popen(["./webapp/run_webapp.sh"], env=_env, stdout=open(TMP / "server.log", "w"),
                          stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(60):
    try:
        urllib.request.urlopen(B + "/health", timeout=2); break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("test server didn't start - see " + str(TMP / "server.log"))

def stop_server():
    try:
        os.killpg(SERVER.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass

class _NoRedir(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, *a, **k):
        return None

def client():
    return urllib.request.build_opener(urllib.request.HTTPCookieProcessor(http.cookiejar.CookieJar()), _NoRedir)

def call(op, method, path, body=None, headers=None, raw=None, ctype="application/json", timeout=120):
    data = raw if raw is not None else (json.dumps(body).encode() if body is not None else None)
    h = {"Content-Type": ctype} if data is not None else {}
    h.update(headers or {})
    req = urllib.request.Request(B + path, data=data, method=method, headers=h)
    try:
        r = op.open(req, timeout=timeout); return r.status, r.read(), dict(r.headers)
    except urllib.error.HTTPError as e:
        loc = e.headers.get("location")
        return e.code, (("-> " + loc).encode() if loc else e.read()), dict(e.headers)
    except Exception as e:
        return 0, str(e).encode(), {}

def js(res):
    try:
        return json.loads(res[1])
    except Exception:
        return {}

RESULTS = []
def check(section, name, ok, detail=""):
    RESULTS.append({"section": section, "check": name, "passed": bool(ok), "detail": str(detail)[:160]})
    print(("PASS " if ok else "FAIL ") + f"[{section}] {name:58s} {str(detail)[:90]}")

def server_tracebacks():
    return (TMP / "server.log").read_text(errors="ignore").count("Traceback")


In [ ]:
try:
    S = "setup"
    admin, par, par2, tut, tut2, anon, dev = client(), client(), client(), client(), client(), client(), client()
    check(S, "first admin can be created on this computer", call(admin, "POST", "/api/auth/setup",
          {"name": "Admin", "email": "admin@test.local", "password": "AdminPass123"})[0] == 200)
    def mk(e, role, pw):
        return js(call(admin, "POST", "/api/users", {"name": e.split("@")[0], "email": e, "role": role, "password": pw}))["user"]["id"]
    p1, p2 = mk("p1@test.local", "parent", "ParentPass9"), mk("p2@test.local", "parent", "ParentPass9")
    t1, t2 = mk("t1@test.local", "tutor", "TutorPass99"), mk("t2@test.local", "tutor", "TutorPass99")
    kid = js(call(admin, "POST", "/api/kids", {"nickname": "Kiddo", "year_level": 2, "parent_id": p1, "tutor_id": t1}))["kid"]["id"]
    for c, e, pw in ((par, "p1", "ParentPass9"), (par2, "p2", "ParentPass9"), (tut, "t1", "TutorPass99"), (tut2, "t2", "TutorPass99")):
        call(c, "POST", "/api/auth/login", {"email": f"{e}@test.local", "password": pw})

    S = "pages"
    check(S, "signed-out visitor sent to sign-in", b"/login.html" in call(anon, "GET", "/")[1])
    check(S, "admin page needs admin", b"/parent.html" in call(par, "GET", "/admin.html")[1])
    check(S, "second admin can't be created", call(anon, "POST", "/api/auth/setup", {"name": "X", "email": "x@x.co", "password": "Abcdefgh123"})[0] == 409)
    check(S, "setup refused through a tunnel", call(client(), "POST", "/api/auth/setup", {"name": "X", "email": "x@x.co", "password": "Abcdefgh123"},
          headers={"X-Forwarded-For": "1.2.3.4"})[0] in (403, 409))

    S = "cross-site"
    check(S, "foreign website can't act as a parent", call(par, "POST", "/api/kids", {"nickname": "csrf"}, headers={"Origin": "https://evil.trycloudflare.com"})[0] == 403)
    check(S, "plain-text form post refused", call(par, "POST", "/api/kids", raw=b'{"nickname":"c2"}', ctype="text/plain", headers={"Origin": "https://evil.example"})[0] == 403)
    check(S, "cross-site request without Origin refused", call(par, "POST", "/api/kids", {"nickname": "c3"}, headers={"Sec-Fetch-Site": "cross-site"})[0] == 403)
    check(S, "(control) GEETA's own page still works", call(par, "POST", "/api/kids", {"nickname": "legit"}, headers={"Origin": B, "Sec-Fetch-Site": "same-origin"})[0] == 200)

    S = "injection"
    call(par, "PATCH", f"/api/kids/{kid}", {"nickname": '<a href="https://evil.example">X</a><img src="/etc/hosts"/>'[:40]})
    sid = js(call(par, "POST", "/api/session", {"kid_id": kid}))["sid"]
    for t in ("what comes next 2, 4, 6, __", "8"):
        call(par, "POST", f"/api/session/{sid}/text", {"text": t})
    rep = js(call(par, "POST", f"/api/session/{sid}/close")).get("report") or {}
    pdf = call(par, "GET", rep.get("pdf_url", "/x"))[1] if rep.get("pdf_url") else b""
    check(S, "markup in a child's name can't break or inject into the PDF", pdf.startswith(b"%PDF") and b"evil.example" not in pdf, f"{len(pdf)} bytes")
    call(par, "PATCH", f"/api/kids/{kid}", {"nickname": "../../../../tmp/geeta_pwned"})
    sid = js(call(par, "POST", "/api/session", {"kid_id": kid}))["sid"]
    for t in ("what comes next 2, 4, 6, __", "8"):
        call(par, "POST", f"/api/session/{sid}/text", {"text": t})
    call(par, "POST", f"/api/session/{sid}/close")
    import glob
    check(S, "child's name can't write files outside the reports folder", not (glob.glob("/tmp/geeta_pwned*") + glob.glob("/private/tmp/geeta_pwned*")))
    call(par, "PATCH", f"/api/kids/{kid}", {"nickname": "Kiddo"})

    S = "size+flood"
    sid = js(call(par, "POST", "/api/session", {"kid_id": kid}))["sid"]
    check(S, "1 MB chat message refused", call(par, "POST", f"/api/session/{sid}/text", {"text": "1 + " * 250000})[0] == 422)
    check(S, "2 MB password refused", call(anon, "POST", "/api/auth/login", {"email": "p1@test.local", "password": "x" * 2_000_000})[0] == 422)
    def png(w, h):
        rawpix = b"".join(b"\x00" + b"\x00" * w for _ in range(h))
        ch = lambda t, d: struct.pack(">I", len(d)) + t + d + struct.pack(">I", zlib.crc32(t + d) & 0xffffffff)
        return b"\x89PNG\r\n\x1a\n" + ch(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 0, 0, 0, 0)) + ch(b"IDAT", zlib.compress(rawpix, 9)) + ch(b"IEND", b"")
    mp = lambda name, data: (f"--XyZ\r\nContent-Disposition: form-data; name=\"file\"; filename=\"{name}\"\r\nContent-Type: image/png\r\n\r\n").encode() + data + b"\r\n--XyZ--\r\n"
    r = call(par, "POST", f"/api/session/{sid}/image", raw=mp("bomb.png", png(30000, 30000)), ctype="multipart/form-data; boundary=XyZ")
    check(S, "900-megapixel decompression-bomb image refused", b"doesn't look like" in r[1])
    r = call(par, "POST", f"/api/session/{sid}/image", raw=mp("x.png", b"#!/bin/sh\necho hi\n" * 50), ctype="multipart/form-data; boundary=XyZ")
    check(S, "non-image file refused as a photo", b"doesn't look like" in r[1])
    sk = socket.create_connection(("127.0.0.1", PORT)); sk.settimeout(5)
    sk.sendall(b"POST /api/session/abc/image HTTP/1.1\r\nHost: 127.0.0.1\r\nContent-Type: multipart/form-data; boundary=x\r\nContent-Length: 209715200\r\n\r\n")
    try:
        first = sk.recv(200).split(b"\r\n")[0]
    except Exception:
        first = b"(waited)"
    check(S, "200 MB upload refused before reading it", b"413" in first, first.decode(errors="ignore"))
    codes = [call(par, "POST", "/api/session", {"kid_id": kid})[0] for _ in range(60)]
    check(S, "session flood limited", 429 in codes, f"{codes.count(200)} allowed")
    print("   waiting 61 s for the session limit to reset..."); time.sleep(61)
    sid = js(call(par, "POST", "/api/session", {"kid_id": kid}))["sid"]
    codes = [call(par, "POST", f"/api/session/{sid}/text", {"text": "hello"})[0] for _ in range(40)]
    check(S, "message flood limited (20 a minute)", codes.count(200) <= 20, f"{codes.count(200)} accepted")
    print("   waiting 61 s for the message limit to reset..."); time.sleep(61)

    S = "guessing"
    codes = [call(anon, "POST", "/api/auth/login", {"email": "p2@test.local", "password": f"g{i}"}, headers={"cf-connecting-ip": f"10.0.{i // 250}.{i % 250}"})[0] for i in range(25)]
    check(S, "password guessing from rotating IPs is locked out", 429 in codes, f"{codes.count(401)} guesses, {codes.count(429)} blocked")
    call(par, "POST", f"/api/kids/{kid}/pin", {"pin": "4826"})
    # guess against a SECOND child, so the lock-out doesn't block the kid checks further down
    kid_g = js(call(admin, "POST", "/api/kids", {"nickname": "Guessme", "year_level": 2, "parent_id": p1, "tutor_id": t1}))["kid"]["id"]
    call(par, "POST", f"/api/kids/{kid_g}/pin", {"pin": "5827"})
    call(dev, "POST", "/api/auth/login", {"email": "p1@test.local", "password": "ParentPass9"}); call(dev, "POST", "/api/kid/device")
    codes = [call(dev, "POST", "/api/kid/login", {"kid_id": kid_g, "pin": f"{i:04d}"}, headers={"cf-connecting-ip": f"10.1.0.{i}"})[0] for i in range(15)]
    check(S, "PIN guessing from rotating IPs is locked out", 429 in codes, f"{codes.count(401)} guesses, {codes.count(429)} blocked")
    check(S, "easy PIN refused", call(par, "POST", f"/api/kids/{kid}/pin", {"pin": "1234"})[0] == 400)

    S = "access"
    check(S, "other parent can't see the child", call(par2, "GET", f"/api/kids/{kid}/progress")[0] == 404)
    check(S, "other parent can't start a session for the child", call(par2, "POST", "/api/session", {"kid_id": kid})[0] == 404)
    check(S, "other parent can't change the child's PIN", call(par2, "POST", f"/api/kids/{kid}/pin", {"pin": "5555"})[0] == 404)
    check(S, "other parent can't delete the child", call(par2, "DELETE", f"/api/kids/{kid}")[0] == 404)
    check(S, "parent can't use admin APIs", all(call(par, "GET", p)[0] == 403 for p in ("/api/users", "/api/admin/overview", "/api/admin/security-log")))
    check(S, "parent can't relink a child", call(par, "PATCH", f"/api/kids/{kid}", {"tutor_id": None})[0] == 403)
    r = call(tut, "POST", "/api/kids", {"nickname": "planted", "parent_id": p2})
    check(S, "tutor can't attach a child to someone's account", not any(k["nickname"] == "planted" for k in js(call(par2, "GET", "/api/kids")).get("kids", [])))
    sid = js(call(par, "POST", "/api/session", {"kid_id": kid}))["sid"]
    check(S, "nobody else can use a live session", call(par2, "POST", f"/api/session/{sid}/text", {"text": "hi"})[0] == 404)

    S = "kid"
    kidc = client()
    call(kidc, "POST", "/api/auth/login", {"email": "p1@test.local", "password": "ParentPass9"}); call(kidc, "POST", "/api/kid/device")
    check(S, "(control) kid signs in with picture + PIN", call(kidc, "POST", "/api/kid/login", {"kid_id": kid, "pin": "4826"})[0] == 200)
    check(S, "kid sign-in signs the grown-up out on that device", call(kidc, "GET", "/api/kids")[0] == 401)
    check(S, "kid can't open grown-up pages", b"/login.html" in call(kidc, "GET", "/parent.html")[1])
    check(S, "kid can't read reports", call(kidc, "GET", "/api/reports/1.pdf")[0] == 401)
    check(S, "kid can't reach the pilot answer key", call(kidc, "GET", "/api/pilot/form")[0] in (401, 403))

    S = "files"
    for p in ("/geeta_server.ipynb", "/run_webapp.sh", "/static/../geeta_server.ipynb", "/static/%2e%2e/geeta_server.ipynb",
              "/static/..%2fgeeta_server.ipynb", "/docs", "/openapi.json", "/static/../../config/credentials.local.yaml",
              "/static/../../data/portal/geeta_portal.db"):
        check(S, f"GET {p} blocked", call(anon, "GET", p)[0] == 404)

    S = "headers"
    c3 = client()
    sc = call(c3, "POST", "/api/auth/login", {"email": "t2@test.local", "password": "TutorPass99"}, headers={"cf-visitor": '{"scheme":"https"}'})[2]
    cookie = (sc.get("set-cookie") or sc.get("Set-Cookie") or "").lower()
    check(S, "sign-in cookie is Secure + HttpOnly + SameSite over https", all(x in cookie for x in ("secure", "httponly", "samesite=lax")))
    h = {k.lower(): v for k, v in call(anon, "GET", "/login.html", headers={"cf-visitor": '{"scheme":"https"}'})[2].items()}
    for name in ("content-security-policy", "x-frame-options", "x-content-type-options", "strict-transport-security",
                 "referrer-policy", "cross-origin-opener-policy"):
        check(S, f"header {name}", name in h)

    S = "2-step"
    mf = client()
    mfid = mk("mfa@test.local", "parent", "ParentPass9")
    call(mf, "POST", "/api/auth/login", {"email": "mfa@test.local", "password": "ParentPass9"})
    import base64, hmac, hashlib
    def totp(sec, step):
        key = base64.b32decode(sec + "=" * (-len(sec) % 8)); hh = hmac.new(key, struct.pack(">Q", step), hashlib.sha1).digest(); o = hh[-1] & 15
        return f"{(struct.unpack('>I', hh[o:o + 4])[0] & 0x7fffffff) % 1000000:06d}"
    sec = js(call(mf, "POST", "/api/auth/2step/start"))["secret"]
    rec = js(call(mf, "POST", "/api/auth/2step/confirm", {"code": totp(sec, int(time.time() // 30))})).get("recovery_codes", [])
    check(S, "(control) 2-step can be switched on", len(rec) == 8)
    att = client(); body = js(call(att, "POST", "/api/auth/login", {"email": "mfa@test.local", "password": "ParentPass9"}))
    check(S, "password alone isn't enough when 2-step is on", body.get("needs_code") and call(att, "GET", "/api/kids")[0] == 401)
    check(S, "made-up pending token refused", call(att, "POST", "/api/auth/login/code", {"pending": "nope", "code": "123456"})[0] == 401)
    ok_c = client(); pend = js(call(ok_c, "POST", "/api/auth/login", {"email": "mfa@test.local", "password": "ParentPass9"}))["pending"]
    check(S, "(control) recovery code works", call(ok_c, "POST", "/api/auth/login/code", {"pending": pend, "code": rec[0]})[0] == 200)
    again = client(); pend2 = js(call(again, "POST", "/api/auth/login", {"email": "mfa@test.local", "password": "ParentPass9"}))["pending"]
    check(S, "recovery code works only once", call(again, "POST", "/api/auth/login/code", {"pending": pend2, "code": rec[0]})[0] == 401)
    check(S, "used pending token can't be reused", call(again, "POST", "/api/auth/login/code", {"pending": pend, "code": rec[1]})[0] == 401)
    codes = [call(again, "POST", "/api/auth/login/code", {"pending": pend2, "code": f"{i:06d}"})[0] for i in range(8)]
    check(S, "2-step code guessing locked out", 429 in codes, str(codes))

    S = "pilot"
    pp, tt = client(), client()
    call(pp, "POST", "/api/auth/login", {"email": "p1@test.local", "password": "ParentPass9"})
    call(tt, "POST", "/api/auth/login", {"email": "t1@test.local", "password": "TutorPass99"})
    check(S, "parent can't read the answer key", call(pp, "GET", "/api/pilot/form")[0] == 403)
    check(S, "parent can't open the printable check", b"/parent.html" in call(pp, "GET", "/pilot-test.html")[1])
    check(S, "parent can't enter scores", call(pp, "POST", f"/api/kids/{kid}/pilot-score", {"phase": "pre", "score": 12, "out_of": 12})[0] == 403)
    check(S, "parent can't set a study code", call(pp, "PATCH", f"/api/kids/{kid}", {"study_code": "P99"})[0] == 403)
    check(S, "parent can't download research data", call(pp, "GET", "/api/pilot/export.csv")[0] == 403)
    check(S, "other tutor can't score someone else's student", call(tut2, "POST", f"/api/kids/{kid}/pilot-score", {"phase": "pre", "score": 1, "out_of": 12})[0] == 404)
    check(S, "non-admin tutor can't download research data", call(tt, "GET", "/api/pilot/export.csv")[0] == 403)
    check(S, "signed-out visitor can't read the answer key", call(anon, "GET", "/api/pilot/form")[0] == 401)
    check(S, "bad study code refused", call(tt, "PATCH", f"/api/kids/{kid}", {"study_code": "p01;drop"})[0] in (400, 422))
    check(S, "impossible score refused", call(tt, "POST", f"/api/kids/{kid}/pilot-score", {"phase": "pre", "score": 99, "out_of": 12})[0] == 400)
    check(S, "unknown phase refused", call(tt, "POST", f"/api/kids/{kid}/pilot-score", {"phase": "mid", "score": 3, "out_of": 12})[0] in (400, 422))
    call(tt, "PATCH", f"/api/kids/{kid}", {"study_code": "P01"})
    call(tt, "POST", f"/api/kids/{kid}/pilot-score", {"phase": "pre", "score": 5, "out_of": 12})
    call(tt, "POST", f"/api/kids/{kid}/pilot-score", {"phase": "post", "score": 8, "out_of": 12})
    csv = call(admin, "GET", "/api/pilot/export.csv")[1].decode()
    check(S, "research export has no names or emails", "P01" in csv and not any(x in csv for x in ("Kiddo", "@test")))
    check(S, "research export gain is right (5/12 -> 8/12 = +25.0)", "25.0" in csv)

    S = "email"
    gone = (("GET", "/api/admin/email"), ("POST", "/api/admin/email/test"), ("POST", "/api/admin/reports/preview"), ("PATCH", "/api/auth/prefs"))
    check(S, "email feature removed: no email endpoints left", all(call(admin, m, p)[0] in (404, 405) for m, p in gone))

    S = "logs"
    ev = js(call(admin, "GET", "/api/admin/security-log")).get("events", [])
    acts = {e["action"] for e in ev}
    check(S, "security log records failures, lock-outs and blocked sites", {"login_failed", "account_locked", "blocked_cross_site", "2step_failed"} <= acts)
    check(S, "security log holds no passwords or codes", not any(x in json.dumps(ev) for x in ("ParentPass9", "TutorPass99", rec[0])))
    check("server", "no server crashes during the attacks", server_tracebacks() == 0, f"{server_tracebacks()} tracebacks")
finally:
    stop_server()

passed = sum(r["passed"] for r in RESULTS)
OUT = ROOT / "outputs" / "tests"; OUT.mkdir(parents=True, exist_ok=True)
json.dump({"when": time.strftime("%Y-%m-%d %H:%M"), "passed": passed, "total": len(RESULTS), "results": RESULTS},
          open(OUT / "security.json", "w"), indent=1)
print(f"\nSECURITY: {passed}/{len(RESULTS)} checks passed")
assert passed == len(RESULTS), "SECURITY SUITE FAILED - see FAIL lines above"
